# Tanka 01: a project from `tk init`

Tanka wraps Jsonnet with a project layout and a notion of *environment*: a directory with
`main.jsonnet` (the workload) and `spec.json` (cluster, namespace, labels). This track builds a
small lab project in `/source/work/tanka-lab` and grows it notebook by notebook.


In [ ]:
export HOME=/tmp
mkdir -p /source/work && cd /source/work && rm -rf tanka-lab && mkdir tanka-lab && cd tanka-lab && tk init && find . -maxdepth 2 -not -path './vendor/*' | sort


In [ ]:
cd /source/work/tanka-lab
cat environments/default/spec.json; echo; cat environments/default/main.jsonnet; echo; cat lib/k.libsonnet; echo; cat jsonnetfile.json | jq '.dependencies[].source.git.subdir'


The context Tanka itself knows about lives in `spec.json`: API server, namespace, labels to inject. Everything else about an environment is plain Jsonnet. Commands that talk to a cluster (`tk env list` once a server is set, `tk diff`, `tk apply`) need a reachable API server and a kubeconfig context for it; the lab has neither, so from here on we stay with the render-only commands: `tk show`, `tk eval`, `tk export`.


In [ ]:
cd /source/work/tanka-lab
tk env list
# tk env set would validate the server against a kubeconfig context; spec.json is plain JSON, so CI usually edits it directly
jq '.spec.apiServer = "https://localhost:6443" | .spec.namespace = "lab" | .spec.injectLabels = true' environments/default/spec.json > /tmp/spec.json && mv /tmp/spec.json environments/default/spec.json && cat environments/default/spec.json


In [ ]:
cd /source/work/tanka-lab
cat > lib/web.libsonnet <<'JSONNET'
local k = import 'k.libsonnet';

// the workload as a function of its context
{
  new(cfg):: {
    local labels = { 'app.kubernetes.io/name': cfg.name, environment: cfg.env },
    local container =
      k.core.v1.container.new(cfg.name, cfg.image)
      + k.core.v1.container.withPorts([k.core.v1.containerPort.newNamed(80, 'http')]),

    deployment:
      k.apps.v1.deployment.new(cfg.name, cfg.replicas, [container], podLabels=labels)
      + k.apps.v1.deployment.metadata.withLabels(labels),

    service:
      k.core.v1.service.new(cfg.name, { 'app.kubernetes.io/name': cfg.name }, [k.core.v1.servicePort.newNamed('http', 80, 'http')])
      + k.core.v1.service.metadata.withLabels(labels),
  },
}
JSONNET
cat > environments/default/main.jsonnet <<'JSONNET'
local web = import 'web.libsonnet';

{
  _config:: { name: 'web', env: 'lab', image: 'traefik/whoami:v1.11.0', replicas: 2 },
  web: web.new($._config),
}
JSONNET
tk show environments/default --dangerous-allow-redirect | head -40


In [ ]:
cd /source/work/tanka-lab
tk eval environments/default | jq '.web.deployment.spec.replicas, .web.deployment.metadata.labels'


In [ ]:
cd /source/work/tanka-lab
rm -rf /tmp/export && tk export /tmp/export environments/default --format '{{.kind}}-{{.metadata.name}}' >/dev/null && ls /tmp/export && grep 'tanka.dev/environment' /tmp/export/Deployment-web.yaml


`tk show` renders, `tk eval` gives the raw Jsonnet value, `tk export` writes files for a GitOps engine. `tk diff` and `tk apply` need a cluster; the rendered manifests pattern does not.
